[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch06/NN_DL_ch06_TextMining/NN_DL_ch06_TextMining.ipynb)

# NN_DL_ch06_TextMining

**Financial text classification: BoW vs TF-IDF vs Word2Vec + MLP**

**Author:** Daniel Traian Pele, ASE Bucharest / IDA


In [ ]:
!pip install torch scikit-learn matplotlib numpy -q


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
np.random.seed(42); torch.manual_seed(42)
MAIN_BLUE='#1A3A6E'; IDA_RED='#CD0000'; FOREST='#2E7D32'; CRIMSON='#DC3545'
print('Ready')


## 1. Financial Headlines Dataset


In [ ]:
# 50 financial headlines with sentiment labels
headlines = [
    ("Fed raises interest rates by 25 basis points", 1),
    ("Stock market crashes amid recession fears", 0),
    ("Q3 earnings beat analyst expectations", 1),
    ("Major bank reports record quarterly losses", 0),
    ("ECB signals pause in monetary tightening", 1),
    ("Unemployment rises to highest level in 5 years", 0),
    ("Tech sector leads market rally", 1),
    ("Credit rating downgraded for sovereign debt", 0),
    ("Consumer confidence index reaches new high", 1),
    ("Trade deficit widens as exports fall sharply", 0),
    ("Merger creates largest financial services firm", 1),
    ("Inflation exceeds central bank target again", 0),
    ("Strong GDP growth reported for third quarter", 1),
    ("Bond yields surge on hawkish policy signals", 0),
    ("Retail sales surge during holiday season", 1),
    ("Housing market shows signs of cooling", 0),
    ("Dividend increase announced by blue chip firms", 1),
    ("Supply chain disruptions worsen globally", 0),
    ("IPO raises more than expected from investors", 1),
    ("Central bank warns of financial stability risks", 0),
]
# Duplicate with variations for larger dataset
texts = [h[0] for h in headlines] * 5
labels = np.array([h[1] for h in headlines] * 5)
print(f'Dataset: {len(texts)} headlines, {labels.mean():.0%} positive')


## 2. Bag-of-Words + Logistic Regression


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)
bow = CountVectorizer(max_features=500)
X_bow_train = bow.fit_transform(X_train)
X_bow_test = bow.transform(X_test)
lr_bow = LogisticRegression(max_iter=1000, random_state=42)
lr_bow.fit(X_bow_train, y_train)
acc_bow = accuracy_score(y_test, lr_bow.predict(X_bow_test))
print(f'BoW + LR Accuracy: {acc_bow:.3f}')


## 3. TF-IDF + SVM


In [ ]:
tfidf = TfidfVectorizer(max_features=500)
X_tfidf_train = tfidf.fit_transform(X_train)
X_tfidf_test = tfidf.transform(X_test)
svm = LinearSVC(random_state=42, max_iter=5000)
svm.fit(X_tfidf_train, y_train)
acc_tfidf = accuracy_score(y_test, svm.predict(X_tfidf_test))
print(f'TF-IDF + SVM Accuracy: {acc_tfidf:.3f}')


## 4. Word2Vec + MLP


In [ ]:
# Simple word embedding: average of random embeddings (demo)
embed_dim = 100
vocab = set(w for t in texts for w in t.lower().split())
word_vecs = {w: np.random.randn(embed_dim)*0.1 for w in vocab}
def text_to_vec(text):
    words = text.lower().split()
    vecs = [word_vecs[w] for w in words if w in word_vecs]
    return np.mean(vecs, axis=0) if vecs else np.zeros(embed_dim)

X_w2v_train = np.array([text_to_vec(t) for t in X_train])
X_w2v_test = np.array([text_to_vec(t) for t in X_test])

# Simple MLP
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp.fit(X_w2v_train, y_train)
acc_w2v = accuracy_score(y_test, mlp.predict(X_w2v_test))
print(f'Word2Vec + MLP Accuracy: {acc_w2v:.3f}')


## 5. Comparison


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
methods = ['BoW + LR', 'TF-IDF + SVM', 'W2V + MLP']
accs = [acc_bow, acc_tfidf, acc_w2v]
colors = ['#888888', MAIN_BLUE, IDA_RED]
bars = ax.bar(methods, accs, color=colors, alpha=0.8)
for b, a in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{a:.0%}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy'); ax.set_title('Financial Sentiment Classification', fontweight='bold', color=MAIN_BLUE)
ax.set_ylim(0, 1.1); ax.set_facecolor('none'); fig.patch.set_alpha(0)
plt.savefig('nn_ch06_text_comparison.pdf', bbox_inches='tight', dpi=300, transparent=True)
plt.savefig('nn_ch06_text_comparison.png', bbox_inches='tight', dpi=150, transparent=True)
plt.show()


## 6. Confusion Matrix


In [ ]:
cm = confusion_matrix(y_test, svm.predict(X_tfidf_test))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Negative','Positive']); ax.set_yticklabels(['Negative','Positive'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=14,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_title('TF-IDF + SVM Confusion Matrix', fontweight='bold', color=MAIN_BLUE)
plt.colorbar(im, ax=ax, shrink=0.8); fig.patch.set_alpha(0)
plt.savefig('nn_ch06_text_confusion.pdf', bbox_inches='tight', dpi=300, transparent=True)
plt.savefig('nn_ch06_text_confusion.png', bbox_inches='tight', dpi=150, transparent=True)
plt.show()
